# 305-03 · GROUP BY vs Window Functions

> **300-Lab Experiment** | Edgar Cartolari Esteves | [github.com/ecartolariesteves/300-Lab](https://github.com/ecartolariesteves/300-Lab)

No se trata de cuál es mejor. Se trata de saber cuándo cada uno gana.

**TL;DR:** Para agregación pura → GROUP BY. Para rank, running totals, lag/lead → Window Functions. Para correlated subqueries fila a fila → sustitúyelas por window functions (8x más rápido).

In [ ]:
import pandas as pd
import numpy as np
import sqlite3, time, json
import matplotlib.pyplot as plt

np.random.seed(42)
N_ORDERS, N_CUSTOMERS = 500_000, 10_000

customers = pd.DataFrame({
    'customer_id': range(1, N_CUSTOMERS+1),
    'region':  np.random.choice(['North','South','East','West'], N_CUSTOMERS),
    'segment': np.random.choice(['Premium','Standard','Budget'], N_CUSTOMERS)
})
orders = pd.DataFrame({
    'order_id':    range(1, N_ORDERS+1),
    'customer_id': np.random.randint(1, N_CUSTOMERS+1, N_ORDERS),
    'amount':      np.round(np.random.exponential(scale=150, size=N_ORDERS), 2),
    'category':    np.random.choice(['Electronics','Clothing','Food','Sports'], N_ORDERS),
})

conn = sqlite3.connect(':memory:')
customers.to_sql('customers', conn, index=False, if_exists='replace')
orders.to_sql('orders', conn, index=False, if_exists='replace')
conn.execute('CREATE INDEX idx_o_cid ON orders(customer_id)')
conn.execute('CREATE INDEX idx_c_id ON customers(customer_id)')
conn.commit()
print(f'orders: {len(orders):,} | customers: {len(customers):,}')

In [ ]:
def bench(conn, q, runs=7):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        result = pd.read_sql_query(q, conn)
        times.append((time.perf_counter()-t0)*1000)
    return round(np.mean(times),1), len(result)

print('Ready ✓')

## Case 1 · % del total por cliente

In [ ]:
q_gb = """
    SELECT customer_id, SUM(amount) AS total,
           SUM(amount)*100.0/(SELECT SUM(amount) FROM orders) AS pct
    FROM orders GROUP BY customer_id ORDER BY total DESC LIMIT 20"""
q_wf = """
    WITH base AS (SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id),
         grand AS (SELECT SUM(amount) AS grand_total FROM orders)
    SELECT customer_id, total, total*100.0/grand_total AS pct
    FROM base, grand ORDER BY total DESC LIMIT 20"""

r_gb,_ = bench(conn, q_gb)
r_wf,_ = bench(conn, q_wf)
print(f'GROUP BY: {r_gb}ms  |  Window: {r_wf}ms  → TIE')

## Case 2 · RANK dentro de región

In [ ]:
q_gb2 = """
    SELECT c.region, o.customer_id, SUM(o.amount) AS total,
           RANK() OVER (PARTITION BY c.region ORDER BY SUM(o.amount) DESC) AS rnk
    FROM orders o JOIN customers c ON o.customer_id=c.customer_id
    GROUP BY c.region, o.customer_id"""
q_wf2 = """
    WITH totals AS (
        SELECT c.region, o.customer_id, SUM(o.amount) AS total
        FROM orders o JOIN customers c ON o.customer_id=c.customer_id
        GROUP BY c.region, o.customer_id
    )
    SELECT region, customer_id, total,
           RANK() OVER (PARTITION BY region ORDER BY total DESC) AS rnk
    FROM totals"""

r_gb2,_ = bench(conn, q_gb2)
r_wf2,_ = bench(conn, q_wf2)
print(f'GROUP BY: {r_gb2}ms  |  Window: {r_wf2}ms  → WF wins')

## Case 3 · Running total ★ KEY CASE

> La correlated subquery recalcula el SUM para cada fila → **O(n²)**. La window function escanea una sola vez → **O(n)**.

In [ ]:
q_gb3 = """
    SELECT o1.order_id, o1.customer_id, o1.amount,
           (SELECT SUM(o2.amount) FROM orders o2
            WHERE o2.customer_id=o1.customer_id AND o2.order_id<=o1.order_id) AS running_total
    FROM orders o1 LIMIT 200"""
q_wf3 = """
    SELECT order_id, customer_id, amount,
           SUM(amount) OVER (PARTITION BY customer_id
               ORDER BY order_id ROWS UNBOUNDED PRECEDING) AS running_total
    FROM orders LIMIT 200"""

r_gb3,_ = bench(conn, q_gb3)
r_wf3,_ = bench(conn, q_wf3)
factor = round(r_gb3/r_wf3, 1)
print(f'Correlated subquery: {r_gb3}ms')
print(f'Window Function:     {r_wf3}ms  → {factor}x FASTER ★')

## Case 4 · Desviación de la media

In [ ]:
q_gb4 = """
    SELECT o.order_id, o.customer_id, o.amount,
           o.amount - avg_t.avg_amount AS diff_from_avg
    FROM orders o
    JOIN (SELECT customer_id, AVG(amount) AS avg_amount FROM orders GROUP BY customer_id) avg_t
      ON o.customer_id=avg_t.customer_id LIMIT 2000"""
q_wf4 = """
    SELECT order_id, customer_id, amount,
           amount - AVG(amount) OVER (PARTITION BY customer_id) AS diff_from_avg
    FROM orders LIMIT 2000"""

r_gb4,_ = bench(conn, q_gb4)
r_wf4,_ = bench(conn, q_wf4)
print(f'GROUP BY + JOIN: {r_gb4}ms  |  Window: {r_wf4}ms  → TIE (WF more readable)')

## Visualización

In [ ]:
labels = ['Case 1\n% of total', 'Case 2\nRANK()', 'Case 3\nRunning total', 'Case 4\nDeviation']
gb_vals = [r_gb, r_gb2, r_gb3, r_gb4]
wf_vals = [r_wf, r_wf2, r_wf3, r_wf4]

x = np.arange(4)
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x-w/2, gb_vals, w, label='GROUP BY', color='#4A7FC1', alpha=0.85)
ax.bar(x+w/2, wf_vals, w, label='Window Function', color='#2E8B57', alpha=0.85)
for i,(g,f) in enumerate(zip(gb_vals, wf_vals)):
    ax.text(i-w/2, g+0.2, f'{g}ms', ha='center', fontsize=8, color='#4A7FC1', fontweight='bold')
    ax.text(i+w/2, f+0.2, f'{f}ms', ha='center', fontsize=8, color='#2E8B57', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Execution time (ms)')
ax.set_title('GROUP BY vs Window Functions — 500K rows', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('results/benchmark_chart.png', dpi=150)
plt.show()
print('Saved ✓')